# 2 · Pose estimation demo

Given the calibrated camera from notebook 1, recover an ArUco marker's 6-DoF pose from a single image and compare every PnP solver.

In [1]:
import os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np, cv2

## Load the camera and estimate a pose

In [2]:
from campose import PoseEstimator

estimator = PoseEstimator.from_calibration('data/calibration_results/sample_calibration.json')
image = cv2.imread('data/sample_aruco_images/aruco_00.png')
pose = estimator.estimate_aruco(image, marker_size=0.08)[0]
print('marker id        :', pose.identifier)
print('translation (m)  :', np.round(pose.translation, 4))
print('euler (deg)      :', np.round(pose.rotation_euler, 2))
print('distance (cm)    :', round(pose.distance * 100, 2))
print('reprojection (px):', round(pose.reprojection_error, 3))

marker id        : 17
translation (m)  : [-0.0129 -0.0272  0.2998]
euler (deg)      : [-170.97    8.7     1.15]
distance (cm)    : 30.13
reprojection (px): 0.031


## Draw the pose axes

The canonical RGB=XYZ axes projected onto the marker are the standard visual proof of a pose estimate.

In [3]:
from campose.visualization import draw_axes
overlay = draw_axes(image, estimator.intrinsics, pose.rvec, pose.tvec, length=0.05)
plt.figure(figsize=(6,5))
plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

## Compare the PnP solvers

Every applicable OpenCV solver on the same correspondences, scored against the marker's true pose.

In [4]:
from campose import evaluation as ev
from campose.pose_estimator import _marker_object_points
from campose.synthetic import render_aruco_marker
from campose.camera_model import CameraIntrinsics

intr = estimator.intrinsics
rendered = render_aruco_marker(intr, (640,480), 17, 0.08,
                               np.array([0.12,-0.16,0.05]), np.array([0.0,0.0,0.5]))
detected = PoseEstimator(intr).estimate_aruco(rendered.image, 0.08)[0]
trials = ev.solver_comparison(_marker_object_points(0.08), detected.image_points,
                              intr, rendered.rvec, rendered.tvec)
print(f"{'solver':>13} {'t_err(mm)':>10} {'r_err(deg)':>11} {'time(ms)':>9}")
for t in trials:
    print(f'{t.solver:>13} {t.translation_error*1000:10.2f} {t.rotation_error:11.2f} {t.solve_time_ms:9.3f}')

       solver  t_err(mm)  r_err(deg)  time(ms)
    iterative       0.70        0.29     0.338
          p3p       0.70        0.29     0.179
         ap3p       0.70        0.29     0.195
         epnp       0.70        0.29     0.446
        sqpnp       0.70        0.28     0.326
  ippe_square       1.03        0.34     0.028
         ippe       1.03        0.34     0.067


The live webcam demo (`python -m campose live ...`) runs exactly this estimator per frame, adding an FPS counter and a reprojection-error quality light.